# Chemical potentials

Chemical potentials determine the formation energy of point defects, as shown in the [Formation energies](./formation_energies.ipynb) tutorial:

$$ \Delta E_f = E_D - E_B + q(\epsilon_{VBM} + \epsilon_F) - \sum_i \Delta n_i \; \mu_i + E_{corr} $$

where $\mu_i$ is the chemical potential of particle $i$. When we compute formation energies, we make a choice of which system (reservoir) exchanges the particles with our target phase. We therefore define a set of chemical potentials, which in `defermi` can be passed as dictionaries or `Chempots` objects, which behave like dictionaries. `Chempots` objects provide additional methods to convert values from absolute to referenced and to convert dictionaries from and to the `pymatgen` format (`{Element object: value}`).

In [21]:
from defermi.chempots.core import Chempots

chempots_dict = {'Sr':-3,'O':-7} # absolute chemical potentials in eV

chempots = Chempots(chempots_dict)
print(chempots['Sr']) # subscriptable like a dict
print(chempots.keys())

chempots.to_pmg_elements()

-3
dict_keys(['Sr', 'O'])


{Element Sr: -3, Element O: -7}

It is often convenient to express the chemical potentials relative to the elemental phase:

$$ \Delta \mu_i = \mu_i - \mu^{\mathrm{ref}}_i $$

Use the `get_absolute` and `get_referenced` functions to change from one convention to the other.

In [22]:
mu_refs = {'Sr':-2,'O':-5}
chempots.get_referenced(mu_refs)

{'Sr': -1, 'O': -2}

A typical use case is to show the formation energies for different sets of chemical potentials (see the [Formation energies](./formation_energies.ipynb) tutorial). We can more easily manipulate a dictionary of sets of chemical potentials with the `Reservoirs` class. Provide the elemental chemical potentials to be used as reference with the `mu_refs` argument. Use the `are_chempots_delta` argument to declare whether you are providing relative (delta) or absolute values.

In [23]:
from defermi.chempots.reservoirs import Reservoirs

res_dict = {
        'A':Chempots({'Sr':-2,'O':-8}),
        'B':Chempots({'Sr':-3,'O':-6.5}),
        'C':Chempots({'Sr':-4,'O':-5})
        }

res = Reservoirs(res_dict=res_dict,mu_refs=mu_refs,are_chempots_delta=False)
res

,Sr,O
A,-2.0,-8.0
B,-3.0,-6.5
C,-4.0,-5.0


Move from absolute to referenced values easily:

In [24]:
res.set_to_referenced()
print(res.are_chempots_delta)
res

True


,Sr,O
A,0.0,-3.0
B,-1.0,-1.5
C,-2.0,0.0


- Filter reservoirs for specific elements
- create latex tables

In [25]:
res.filter_reservoirs(elements=['O'])

,O
A,-3.0
B,-1.5
C,0.0


In [26]:
print(res.get_latex_table())

\begin{tabular}{lrr}
\toprule
 & $\Delta \mu_{\text{Sr}}$ & $\Delta \mu_{\text{O}}$ \\
\midrule
A & 0.000000 & -3.000000 \\
B & -1.000000 & -1.500000 \\
C & -2.000000 & 0.000000 \\
\bottomrule
\end{tabular}



## Phase diagram

Reservoirs are often chosen from the stability region of the target phase, obtained from the phase diagram. We can store the phase diagram in the `Reservoirs` object.  Phase diagrams computed with DFT can be also pulled from the [Materials Project](https://next-gen.materialsproject.org/). `defermi` provides wrapper functions for the MP API in the `tools.materials_project` module. You 

In [27]:
from defermi.tools.materials_project import MPDatabase
from pymatgen.core.composition import Composition

phase_diagram = MPDatabase().get_phase_diagram_from_chemsys('Ba-Ti-O') #if not stored in your .pmgrc file, provide the MP API_KEY
phase_diagram

Ba-Ti-O phase diagram
21 stable phases: 
TiO, BaO2, Ti3O5, BaO10, Ti2O, Ba, O2, BaO, Ti3O, Ba2Ti13O22, Ba4Ti13O30, Ba2TiO4, Ti2O3, BaTiO3, Ba3Ti20O40, Ti6O, TiO2, BaTi2O5, Ti, BaTi8O16, Ba2Ti6O13

In [28]:
boundaries = phase_diagram.get_all_chempots(Composition('BaTiO3'))
res_dict = {key: Chempots.from_pmg_elements(value) for key,value in boundaries.items()}
res = Reservoirs(res_dict,phase_diagram=phase_diagram,are_chempots_delta=False)
res

,Ba,Ti,O
BaTiO3-BaTi2O5-Ba2Ti6O13,-3.755598,-9.033464,-9.776825
BaTiO3-TiO-Ba2TiO4,-2.938610,-8.398586,-10.260780
BaTiO3-BaO2-Ba2TiO4,-8.073304,-18.667975,-5.126085
BaTiO3-BaO10-BaO2,-8.402881,-18.832763,-4.961297
BaTiO3-BaO10-O2,-8.536242,-18.739409,-4.947961
BaTiO3-BaTi2O5-O2,-8.584461,-18.691191,-4.947961
Ba2Ti13O22-BaTiO3-TiO,-3.304013,-8.581288,-10.078078
Ba2Ti13O22-BaTiO3-Ba2Ti6O13,-3.696412,-8.950604,-9.824173


We can use the functions in the `chempots.generator` module to speed-up with workflow.

In [29]:
from defermi.chempots.generator import generate_chempots_from_mp

res = generate_chempots_from_mp(composition='BaTiO3')
res

Pulling chemical potentials from Materials Project database...
Chemical potentials:
                 Ba         Ti          O
O-poor   -2.938610  -8.398586 -10.260780
O-middle -5.737426 -13.568998  -7.604370
O-rich   -8.536242 -18.739409  -4.947961


,Ba,Ti,O
O-poor,-2.938610,-8.398586,-10.260780
O-middle,-5.737426,-13.568998,-7.604370
O-rich,-8.536242,-18.739409,-4.947961
